# Predicción de Demanda de Productos Bancarios

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Proyectar demanda** de productos bancarios con `SNOWFLAKE.ML.FORECAST`
2. **Analizar estacionalidad** en contrataciones
3. **Forecast multi-serie** por producto y canal
4. **Evaluar accuracy** del modelo de forecast
5. **Construir dashboard** de planificación de demanda

---

## Lo Que Construirás

Un **sistema de predicción de demanda** que proyecta contrataciones:
- Forecast a 12 meses por producto y canal
- Detección de patrones estacionales
- Intervalos de confianza para planificación
- Dashboard de forecast vs real

**Valor de Negocio**: Optimizar recursos y stock de productos anticipando la demanda.

---

## Funcionalidades Clave

- `SNOWFLAKE.ML.FORECAST` — Proyección de series temporales
- `GENERATOR()` — Datos históricos sintéticos
- Window functions — Medias móviles y tendencias

¡Comencemos!

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS DEMANDA_PRODUCTOS_DB;
CREATE SCHEMA IF NOT EXISTS DEMANDA_PRODUCTOS_DB.PUBLIC;
USE SCHEMA DEMANDA_PRODUCTOS_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() AS db;

---

## Paso 2: Generar Datos Históricos de Contrataciones

### Qué Vamos a Crear

- **36 meses** de datos de contrataciones
- **6 productos**: Hipotecas, Préstamos, Fondos, Seguros, Tarjetas, Cuentas
- **4 canales**: Oficina, App, Web, Teléfono
- **Estacionalidad** incorporada (más hipotecas en primavera, más seguros en otoño)

In [ ]:
CREATE OR REPLACE TABLE CONTRATACIONES_HISTORICAS (
    fecha DATE,
    producto VARCHAR(50),
    canal VARCHAR(20),
    num_contrataciones INTEGER,
    volumen_eur DECIMAL(15,2),
    PRIMARY KEY (fecha, producto, canal)
);

INSERT INTO CONTRATACIONES_HISTORICAS
WITH meses AS (
    SELECT DATE_TRUNC('month', DATEADD('month', -SEQ4(), CURRENT_DATE())) AS fecha
    FROM TABLE(GENERATOR(ROWCOUNT => 36))
),
prods AS (SELECT column1 AS producto FROM VALUES ('Hipotecas'),('Préstamos'),('Fondos'),('Seguros'),('Tarjetas'),('Cuentas')),
canales AS (SELECT column1 AS canal FROM VALUES ('Oficina'),('App'),('Web'),('Teléfono'))
SELECT
    m.fecha, p.producto, c.canal,
    GREATEST(1, UNIFORM(20, 200, RANDOM()) +
        CASE WHEN p.producto = 'Hipotecas' AND MONTH(m.fecha) IN (3,4,5) THEN 50
             WHEN p.producto = 'Seguros' AND MONTH(m.fecha) IN (9,10,11) THEN 40
             WHEN p.producto = 'Tarjetas' AND MONTH(m.fecha) IN (11,12) THEN 60
             ELSE 0 END +
        CASE WHEN c.canal = 'App' THEN UNIFORM(10, 30, RANDOM()) ELSE 0 END
    ),
    ROUND(UNIFORM(100000, 5000000, RANDOM()), 2)
FROM meses m CROSS JOIN prods p CROSS JOIN canales c;

SELECT producto, SUM(num_contrataciones) AS total FROM CONTRATACIONES_HISTORICAS GROUP BY 1 ORDER BY 2 DESC;

---

## Paso 3: Entrenar Modelo de Forecast

### `SNOWFLAKE.ML.FORECAST` Multi-Serie

Proyectamos contrataciones por producto (series independientes).

### ¿Por Qué ML.FORECAST?

- **Multi-serie**: Cada producto tiene su propio patrón
- **Estacionalidad automática**: Detecta ciclos mensuales/anuales
- **Intervalos de confianza**: Rangos para planificación conservadora

In [ ]:
-- Agregar por producto para forecast
CREATE OR REPLACE TABLE DEMANDA_POR_PRODUCTO AS
SELECT
    fecha,
    producto,
    SUM(num_contrataciones) AS contrataciones_total
FROM CONTRATACIONES_HISTORICAS
GROUP BY 1, 2;

-- Entrenar modelo de forecast
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST forecast_demanda(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'DEMANDA_POR_PRODUCTO'),
    TIMESTAMP_COLNAME => 'FECHA',
    TARGET_COLNAME => 'CONTRATACIONES_TOTAL',
    SERIES_COLNAME => 'PRODUCTO'
);

-- Proyectar 12 meses
CALL forecast_demanda!FORECAST(FORECASTING_PERIODS => 12);

---

## Paso 4: Analizar Resultados del Forecast

### Evaluación

Comparamos proyección con datos recientes para validar la precisión.

In [ ]:
-- Guardar y analizar forecast
CREATE OR REPLACE TABLE FORECAST_DEMANDA AS
SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

-- Resumen por producto
SELECT 
    SERIES::VARCHAR AS producto,
    ROUND(AVG(FORECAST), 0) AS forecast_medio_mensual,
    ROUND(MIN(LOWER_BOUND), 0) AS bound_inferior,
    ROUND(MAX(UPPER_BOUND), 0) AS bound_superior
FROM FORECAST_DEMANDA
GROUP BY 1 ORDER BY forecast_medio_mensual DESC;

---

## Paso 5: Dashboard de Planificación

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Predicción de Demanda de Productos')
st.markdown('### Forecast a 12 Meses')

df_hist = session.sql('SELECT producto, fecha, contrataciones_total FROM DEMANDA_POR_PRODUCTO ORDER BY fecha').to_pandas()
df_fc = session.sql('SELECT SERIES::VARCHAR AS producto, TS AS fecha, FORECAST AS contrataciones FROM FORECAST_DEMANDA').to_pandas()

producto = st.selectbox('Producto', df_hist['PRODUCTO'].unique())

st.subheader(f'Histórico + Forecast: {producto}')
hist = df_hist[df_hist['PRODUCTO'] == producto].set_index('FECHA')['CONTRATACIONES_TOTAL']
fc = df_fc[df_fc['PRODUCTO'] == producto].set_index('FECHA')['CONTRATACIONES']
combined = pd.DataFrame({'Histórico': hist, 'Forecast': fc})
st.line_chart(combined)

st.markdown('---')
st.subheader('Resumen por Producto')
df_res = session.sql("SELECT SERIES::VARCHAR AS producto, ROUND(SUM(FORECAST),0) AS total_forecast_12m FROM FORECAST_DEMANDA GROUP BY 1 ORDER BY 2 DESC").to_pandas()
st.dataframe(df_res)

---

## Paso 6: Limpieza (Opcional)

In [ ]:
-- Descomentar para limpiar

-- DROP DATABASE IF EXISTS DEMANDA_PRODUCTOS_DB;